In [40]:
import sqlite3
import pandas as pd
if __name__ != '__main__':
    quit()

conn = sqlite3.connect('../data/checking-logs.sqlite') 

In [41]:

schema = pd.io.sql.read_sql("PRAGMA table_info(test);", conn)
print("Схема таблицы test:")
print(schema)

Схема таблицы test:
   cid             name       type  notnull dflt_value  pk
0    0              uid       TEXT        0       None   0
1    1          labname       TEXT        0       None   0
2    2  first_commit_ts  TIMESTAMP        0       None   0
3    3    first_view_ts  TIMESTAMP        0       None   0


In [42]:
first_10_rows = pd.io.sql.read_sql("SELECT * FROM test LIMIT 10;", conn)
print("\nПервые 10 строк таблицы test:")
print(first_10_rows)


Первые 10 строк таблицы test:
       uid   labname             first_commit_ts               first_view_ts
0   user_1    laba04  2020-04-26 17:06:18.462708  2020-04-26 21:53:59.624136
1   user_1   laba04s  2020-04-26 17:12:11.843671  2020-04-26 21:53:59.624136
2   user_1    laba05  2020-05-02 19:15:18.540185  2020-04-26 21:53:59.624136
3   user_1    laba06  2020-05-17 16:26:35.268534  2020-04-26 21:53:59.624136
4   user_1   laba06s  2020-05-20 12:23:37.289724  2020-04-26 21:53:59.624136
5   user_1  project1  2020-05-14 20:56:08.898880  2020-04-26 21:53:59.624136
6  user_10    laba04  2020-04-25 08:24:52.696624  2020-04-18 12:19:50.182714
7  user_10   laba04s  2020-04-25 08:37:54.604222  2020-04-18 12:19:50.182714
8  user_10    laba05  2020-05-01 19:27:26.063245  2020-04-18 12:19:50.182714
9  user_10    laba06  2020-05-19 11:39:28.885637  2020-04-18 12:19:50.182714


In [43]:
min_diff_query = """
SELECT 
    t.uid,
    MIN((julianday(datetime(d.deadlines, 'unixepoch')) - julianday(t.first_commit_ts)) * 24) AS min_diff_hours
FROM 
    test t
JOIN 
    deadlines d ON t.labname = d.labs
WHERE 
    t.labname != 'project1'
GROUP BY 
    t.uid
ORDER BY 
    min_diff_hours ASC
LIMIT 1
"""
test_query = """
SELECT 
    datetime(d.deadlines, 'unixepoch') 
FROM
    deadlines d
"""
df_min = pd.io.sql.read_sql(min_diff_query, conn)
print("\nМинимальная разница (часы):")
print(df_min)



Минимальная разница (часы):
       uid  min_diff_hours
0  user_25        2.867236


In [44]:
max_diff_query = """
SELECT 
    t.uid,
    MAX((julianday(datetime(d.deadlines, 'unixepoch')) - julianday(t.first_commit_ts)) * 24) AS max_diff_hours
FROM 
    test t
JOIN 
    deadlines d ON t.labname = d.labs
WHERE 
    t.labname != 'project1'
GROUP BY 
    t.uid
ORDER BY 
    max_diff_hours DESC
LIMIT 1
"""
df_max = pd.io.sql.read_sql(max_diff_query, conn)
print("\nМаксимальная разница (часы):")
print(df_max)



Максимальная разница (часы):
       uid  max_diff_hours
0  user_30       202.38473


In [45]:
avg_diff_query = """
SELECT 
    AVG((julianday(datetime(d.deadlines, 'unixepoch')) - julianday(t.first_commit_ts)) * 24) AS avg_diff_hours
FROM 
    test t
JOIN 
    deadlines d ON t.labname = d.labs
WHERE 
    t.labname != 'project1'
"""
df_avg = pd.io.sql.read_sql(avg_diff_query, conn)
print("\nСредняя разница (часы):")
print(df_avg)


Средняя разница (часы):
   avg_diff_hours
0       89.687686


In [46]:
correlation_query = """
SELECT 
    t.uid,
    AVG((julianday(datetime(d.deadlines, 'unixepoch')) - julianday(t.first_commit_ts)) * 24) AS avg_diff,
    COUNT(p.uid) AS pageviews
FROM 
    test t
JOIN 
    deadlines d ON t.labname = d.labs
LEFT JOIN 
    pageviews p ON t.uid = p.uid
WHERE 
    t.labname != 'project1'
GROUP BY 
    t.uid
"""
views_diff = pd.io.sql.read_sql(correlation_query, conn)
print(views_diff)

        uid    avg_diff  pageviews
0    user_1   65.119644        140
1   user_10   75.242310        445
2   user_14  159.568696        429
3   user_17   62.207513        235
4   user_18    6.367907          9
5   user_19   99.440298         64
6   user_21   96.111041         40
7   user_25   93.474751        895
8   user_28   86.793652        745
9    user_3  105.738041       1585
10  user_30  145.528546         12


In [47]:
correlation = views_diff['avg_diff'].corr(views_diff['pageviews'])
print(f"\nКоэффициент корреляции между avg_diff и pageviews: {correlation}")



Коэффициент корреляции между avg_diff и pageviews: 0.18504199436324809


In [48]:

conn.close()
